# Color Percentage Analyzer - Interactive Colab Notebook

This notebook allows you to upload any image (via a file selector or drag-and-drop) and get an instant color percentage analysis table and horizontal bar chart.

### How to run:
1. Run **Step 1** to install required libraries.
2. Run **Step 2** to define the analyzer logic.
3. Run **Step 3** to launch the upload widget. You can drag and drop your images there.

### Step 1: Install Dependencies

In [ ]:
!pip install Pillow matplotlib numpy

### Step 2: Define Color Mapping and Analysis Logic

In [ ]:
import sys
import os
import colorsys
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt

def classify_hsv_advanced(h, s, v):
    # 1. Achromatic checks
    if s < 0.08 and v >= 0.85:
        return "White"
    elif v < 0.15:
        return "Black"
    elif s < 0.15 and 0.15 <= v < 0.85:
        return "Gray"
    
    # 2. Chromatic checks
    if h < 15 or h >= 345:
        return "Red"
    elif 15 <= h < 35:
        if v < 0.55:
            return "Brown"
        return "Orange"
    elif 35 <= h < 65:
        if v < 0.50 or s < 0.30:
            return "Brown"
        return "Yellow"
    elif 65 <= h < 165:
        return "Green"
    elif 165 <= h < 265:
        return "Blue"
    elif 265 <= h < 290:
        return "Purple"
    elif 290 <= h < 345:
        if v >= 0.50 and s >= 0.20:
            return "Pink"
        else:
            return "Brown"
    return "Unknown"

def generate_and_show_chart(counts, total_pixels):
    sorted_data = counts.most_common()
    if not sorted_data:
        return
    
    categories = [item[0] for item in sorted_data]
    percentages = [(item[1] / total_pixels) * 100 for item in sorted_data]
    
    hex_colors = {
        "White": "#F8F9FA",
        "Black": "#212529",
        "Gray": "#6C757D",
        "Red": "#DC3545",
        "Orange": "#FD7E14",
        "Yellow": "#FFC107",
        "Green": "#198754",
        "Blue": "#0D6EFD",
        "Purple": "#6F42C1",
        "Pink": "#E83E8C",
        "Brown": "#795548",
    }
    
    colors = [hex_colors.get(cat, "#333333") for cat in categories]
    edge_colors = ["#CED4DA" if cat == "White" else "none" for cat in categories]
    
    fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
    fig.patch.set_facecolor('#F8F9FA')
    ax.set_facecolor('#F8F9FA')
    
    bars = ax.barh(categories, percentages, color=colors, edgecolor=edge_colors, height=0.6, linewidth=1)
    ax.invert_yaxis()
    ax.set_xlabel('Percentage Share (%)', fontsize=12, fontweight='bold', color='#495057')
    ax.set_title('Color Percentage Breakdown', fontsize=16, fontweight='bold', color='#212529', pad=15)
    ax.set_xlim(0, max(percentages) * 1.15)
    
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color('#DEE2E6')
    ax.tick_params(axis='y', left=False, labelsize=11)
    ax.tick_params(axis='x', labelsize=10, colors='#6C757D')
    
    for bar, pct in zip(bars, percentages):
        width = bar.get_width()
        ax.text(width + 0.8, bar.get_y() + bar.get_height()/2, f'{pct:.1f}%',
                va='center', ha='left', fontsize=10, fontweight='bold', color='#495057')
    plt.tight_layout()
    plt.show()

def analyze_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                img = img.convert('RGBA')
                background = Image.new("RGBA", img.size, (255, 255, 255, 255))
                img = Image.alpha_composite(background, img)
                img = img.convert('RGB')
            elif img.mode != 'RGB':
                img = img.convert('RGB')
            
            width, height = img.size
            total_raw_pixels = width * height
            
            if total_raw_pixels > 1_000_000:
                scale_factor = min(800.0 / width, 800.0 / height)
                new_size = (int(width * scale_factor), int(height * scale_factor))
                img = img.resize(new_size, Image.Resampling.LANCZOS)
                width, height = img.size
            
            pixel_access = img.load()
            pixels = []
            for y in range(height):
                for x in range(width):
                    pixels.append(pixel_access[x, y])
    except Exception as e:
        print(f"Error: {e}")
        return
    
    total_pixels = len(pixels)
    categories = []
    for r, g, b in pixels:
        h_norm, s, v = colorsys.rgb_to_hsv(r / 255.0, g / 255.0, b / 255.0)
        h = h_norm * 360.0
        category = classify_hsv_advanced(h, s, v)
        categories.append(category)
    
    counts = Counter(categories)
    
    print(f"\nTotal Pixels Scanned: {total_pixels:,}")
    print("|" + "-"*17 + "|" + "-"*12 + "|" + "-"*14 + "|")
    print(f"| {'Color Category':<15} | {'Percentage':<10} | {'Pixel Count':<12} |")
    print("|" + "-"*17 + "|" + "-"*12 + "|" + "-"*14 + "|")
    for category, count in counts.most_common():
        percentage = (count / total_pixels) * 100
        print(f"| {category:<15} | {percentage:>8.2f}% | {count:>12,} |")
    
    generate_and_show_chart(counts, total_pixels)

### Step 3: Drag & Drop / Upload Image for Analysis

In [ ]:
from google.colab import files

print("Upload your image below (Click 'Choose Files' or Drag-and-Drop your image file):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n--- ANALYZING: {filename} ---")
    analyze_image(filename)